In [ ]:
from mascope_lib.file_func import get_instrument_type, load_file, get_sum_signal
from mascope_lib.peak import fit_n_peaks, gen_peak
import numpy as np
import asyncio
import os
from concurrent.futures import ProcessPoolExecutor
import json
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import matplotlib.pyplot as plt
from ipywidgets import IntSlider, interact, VBox
from scipy.stats import pearsonr

In [ ]:
def segment_spec(sum_spec):
    """Perform segmentation of Orbitrap spectrum

    :param sum_spec: sum of spectra along time dimension
    :type sum_spec: array-like
    :return: list of segment indices
    :rtype: list
    """
    # Get non-zero indices
    non_zero_indices = np.flatnonzero(sum_spec)
    # Split in chunks taking into account repeating zeros
    non_zero_indices = np.split(
        non_zero_indices, np.where(np.diff(non_zero_indices) > 2)[0] + 1
    )
    # Add zeros on chunk borders
    non_zero_indices = [
        np.concatenate(([chunk[0] - 1], chunk, [chunk[-1] + 1]))
        for chunk in non_zero_indices
    ]
    # Check wrong indices on the spectrum ends
    if non_zero_indices[0][0] < 0:
        # Remove negative index in the first chunk
        non_zero_indices[0] = non_zero_indices[0][1:]
    if non_zero_indices[-1][-1] == len(sum_spec):
        # Remove out-of-range index from the last chunk
        non_zero_indices[-1][-1] = non_zero_indices[-1][:-1]
    # Remove short chunks
    non_zero_indices = [chunk for chunk in non_zero_indices if len(chunk) > 4]
    return non_zero_indices

In [ ]:
def get_resolution_function(mz, instrument_type):
    """Calculates the value of the resolution function"""

    if instrument_type == "tof":
        R0 = 8190.820
        m0 = 39.264
        dm = 12.641
        return R0 - R0 / (1 + np.exp((mz - m0) / dm))
    if instrument_type == "orbi":
        return 1.725e6 / np.sqrt(mz)
    raise Exception("Unknown instrument type")


def get_peak_shape(instrument_type):
    """Get peak shape as dict"""
    with open(
        f"C:/mascope_data/peak_shapes/peak_shape_{instrument_type}.json", "r"
    ) as f:
        peak_shape = json.load(f)

    return peak_shape

In [ ]:
async def fit_time_sweep(mzs, specs_2D, time_span=None):
    cpu_cores = os.cpu_count()
    max_workers = max(1, cpu_cores // 2)
    executor = ProcessPoolExecutor(max_workers=max_workers)
    loop = asyncio.get_event_loop()

    if time_span == None:
        spec_to_fit = list(zip(mzs, specs_2D))
    else:
        spec_to_fit = list(zip(mzs, [array[time_span] for array in specs_2D]))
    # Fill in asynchronous operations
    futures = [
        loop.run_in_executor(
            executor,
            fit_n_peaks,
            mz_to_fit,
            spec_to_fit,
            peak_shape,
            get_resolution_function(mz_to_fit.mean(), instrument_type),
            0.8,
            5,
        )
        for mz_to_fit, spec_to_fit in spec_to_fit
    ]

    new_peaks = []
    for i, future in enumerate(asyncio.as_completed(futures)):
        fit, peaks = await future

        if fit:
            new_peaks.extend(peaks)

        print(f"Peak detection progress: {(i+1)/len(futures):.2f}")

    executor.shutdown()

    new_peak_mzs, new_peak_heights, new_peak_areas = [], [], []
    if len(new_peaks) > 0:
        new_peak_mzs = list(zip(*new_peaks))[0]
        new_peak_heights = list(zip(*new_peaks))[1]
        new_peak_areas = list(zip(*new_peaks))[3]

    return new_peak_mzs, new_peak_heights, new_peak_areas

In [ ]:
filename = "KLTOF1_2024.02.29_15h22m28s"
instrument_type = get_instrument_type(filename)

# Get peak shape
peak_shape = get_peak_shape(instrument_type)

# get 2D signal
file = load_file(filename, vars=["signal"])
signal_2D = file.signal.interpolate_na(dim="mz", method="linear").values

# Get sum spectrum and associated mz
sum_spec = get_sum_signal(filename)
mz = sum_spec.mz

if instrument_type == "orbi":
    # Convert sum_spec and mz to numpy arrays for speed
    mz = mz.values
    sum_spec = sum_spec.values
    # Segment sum spectrum and mz
    non_zero_indices = segment_spec(sum_spec)
    sum_spec_to_fit = [sum_spec[chunk] for chunk in non_zero_indices]
    mzs_to_fit = [mz[chunk] for chunk in non_zero_indices]

    # Slice each 2D signal based on non_zero_indices
    specs_to_fit_2D = []
    for val in signal_2D.T:
        specs_to_fit_2D.append([val[chunk] for chunk in non_zero_indices])

if instrument_type == "tof":
    # MZ window width
    dmz = 0.5
    # Integer MZ values to fit
    u_list = np.arange(10, np.floor(mz.values.max()) + 1)
    # Segment sum spectrum and related mz scale
    sum_spec_to_fit = [
        sum_spec.sel(mz=slice(u - dmz, u + dmz)).compute().values for u in u_list
    ]
    mzs_to_fit = [mz.sel(mz=slice(u - dmz, u + dmz)).compute().values for u in u_list]
    # Convert mz to numpy array
    mz = mz.values

    # Slice each 2D signal based on non_zero_indices
    specs_to_fit_2D = []
    for val in signal_2D.T:
        spec_arr = []
        for u in u_list:
            mz_mask = np.where((mz >= (u - dmz)) & (mz <= (u + dmz)))
            spec_arr.append(val[mz_mask])

        specs_to_fit_2D.append(spec_arr)

ncols = len(specs_to_fit_2D[0])
## Block to remove noize slices
specs_to_fit_2D_filt = []
mzs_to_fit_filt = []
sum_spec_to_fit_filt = []
for col in range(ncols):
    # Gather chunks from all time spans
    slc = np.array([row[col] for row in specs_to_fit_2D])
    # If any row in a slc is 0, we drop the entire chunk
    # Set != 1 to filter out chunks having 1 non-zero row only
    if len(slc[~np.all(slc == 0, axis=1)]) == 1:
        continue
    # Add good slice, corresponding sum_spec slice and mz
    specs_to_fit_2D_filt.append(slc)
    mzs_to_fit_filt.append(mzs_to_fit[col])
    sum_spec_to_fit_filt.append(sum_spec_to_fit[col])

time_stamps = file.signal.time.values
print(f"Number of time sweeps: {len(time_stamps)}")
print(f"Number of slices to fit: {len(specs_to_fit_2D_filt)}")

In [ ]:
# Fit all time spans and add them to fitted_time_sweeps list
fitted_time_sweeps = []

for i, val in enumerate(time_stamps):

    fitted_time_sweeps.append(
        await fit_time_sweep(mzs_to_fit_filt, specs_to_fit_2D_filt, i)
    )

In [ ]:
# Fit sum spectrum for comparison
sum_peak_mzs, sum_peak_heights, sum_peak_areas = await fit_time_sweep(
    mzs_to_fit_filt, sum_spec_to_fit_filt
)

In [ ]:
def flatten_list(list2D):
    """Flattens 2D list (what a surprise)"""
    return np.array([item for row in list2D for item in row])


fig = go.Figure(
    [
        go.Scatter(
            x=mz,
            y=sum_spec,
            name="Sum spectrum",
        ),
        go.Scatter(
            x=flatten_list(mzs_to_fit_filt),
            y=flatten_list(sum_spec_to_fit_filt),
            name="Filtered sum spectrum",
        ),
    ]
)

# fig.update_yaxes(range=[-10, 300])
fig.show()

In [ ]:
from operator import itemgetter

# Convert time sweeps to dict for clearance and sort items
time_dict = {}
for i, val in enumerate(fitted_time_sweeps):
    combined_tuples = list(zip(val[0], val[1], val[2]))
    sorted_tuples = sorted(combined_tuples, key=itemgetter(0))
    time_dict[i] = {
        "mz": np.array([t[0] for t in sorted_tuples]),
        "hei": np.array([t[1] for t in sorted_tuples]),
        "area": np.array([t[2] for t in sorted_tuples]),
    }

In [ ]:
def get_peak_areas(time_dict, target_mz, precision=2):
    areas = []
    for key in time_dict.keys():
        mz_diff = np.abs(time_dict[key]["mz"] - target_mz)
        ppm_diff = 1e6 * mz_diff / target_mz
        areas.append(np.sum(time_dict[key]["area"][np.where(ppm_diff < precision)]))

    return areas


def get_peak_heis(time_dict, target_mz, precision=2):
    areas = []
    mzs = []
    for key in time_dict.keys():
        mz_diff = np.abs(time_dict[key]["mz"] - target_mz)
        ppm_diff = 1e6 * mz_diff / target_mz
        areas.append(np.sum(time_dict[key]["area"][np.where(ppm_diff < precision)]))
        mzs.append(np.mean(time_dict[key]["mz"][np.where(ppm_diff < precision)]))
        print(areas[-1], mzs[-1])
    heis = np.array(areas) / (mzs / get_resolution_function(mzs, instrument_type))
    return heis


def get_peak_heis(time_dict, target_mz, precision=2):
    heis = []
    # mzs = []
    for key in time_dict.keys():
        mz_diff = np.abs(time_dict[key]["mz"] - target_mz)
        ppm_diff = 1e6 * mz_diff / target_mz
        heis.append(np.sum(time_dict[key]["hei"][np.where(ppm_diff < precision)]))
        # mzs.append(np.mean(time_dict[key]["mz"][np.where(ppm_diff < precision)]))
    return np.array(heis)


def get_peak_profile(file, sum_peak_mzs, target_mz):
    # Convert sum_peaks_mzs to numpy array
    sum_peak_mzs = np.array(sorted(sum_peak_mzs))
    # Find mz value from fitted sum peaks closest to the target
    abs_diff = np.abs(sum_peak_mzs - target_mz)
    nearest_idx = np.argmin(abs_diff)
    nearest_mz = sum_peak_mzs[nearest_idx]
    # Get the profile of counts change with time span
    # the counts of the mz point closest to nearest_mz is taken from each time span
    spec_signal = file.signal.interpolate_na(dim="mz", method="linear")
    peak_profiles = spec_signal.sel(mz=nearest_mz, method="nearest").values
    return peak_profiles

In [ ]:
# Choose target mzs to check
target_mzs = [
    342.087,
    328.026,
    189.130,
    96.047,
    74.069,
    126.084,
    167.135,
    189.130,
    209.033,
    288.027,
    325.068,
    369.044,
    417.287,
    442.103,
]
# Sort and convert to numpy array
target_mzs = sorted(np.array(target_mzs))
# Choose ppm precision
precision = 40

# Initial target mz index
target_mz_num = 0

# Filter out spectra ranges containing target mzs
target_range_inds = [
    i
    for i, val in enumerate(mzs_to_fit_filt)
    if np.any((target_mzs <= val.max()) & (target_mzs >= val.min()))
]


# Create initial traces
def create_traces(target_mz_num):
    areas = get_peak_areas(time_dict, target_mzs[target_mz_num], precision=precision)
    heis = get_peak_heis(time_dict, target_mzs[target_mz_num], precision=precision)
    heis_old_fashioned = get_peak_profile(file, sum_peak_mzs, target_mzs[target_mz_num])

    sum_trace = go.Scatter(
        x=flatten_list(mzs_to_fit_filt),
        y=flatten_list(sum_spec_to_fit_filt) / len(time_dict.keys()),
        showlegend=False,
    )
    area_trace = go.Scatter(
        x=time_stamps,
        y=heis,
        name="Sum peak height",
        customdata=np.arange(0, len(time_stamps)),
        hovertemplate="Span: %{customdata} | X: %{x:.2f}",
        marker_color="black",
    )
    peak_profile_trace = go.Scatter(
        x=time_stamps,
        y=heis_old_fashioned,
        name="Peak profile",
        customdata=np.arange(0, len(time_stamps)),
        hovertemplate="Span: %{customdata} | X: %{x:.2f}",
        marker_color="grey",
    )
    time_span_traces = [
        go.Scatter(
            x=mzs_to_fit_filt[target_range_inds[target_mz_num]],
            y=specs_to_fit_2D_filt[target_range_inds[target_mz_num]][i],
            name=f"Time span {i}",
        )
        for i in time_dict.keys()
    ]

    return sum_trace, area_trace, peak_profile_trace, time_span_traces


sum_trace, area_trace, peak_profile_trace, time_span_traces = create_traces(
    target_mz_num
)

# Create subplots
fig = make_subplots(
    rows=2,
    cols=2,
    specs=[[{"colspan": 2}, None], [{}, {}]],
    subplot_titles=("Sum spectrum", "Time evolution", "Time spans"),
)

# Add initial traces
fig.add_trace(sum_trace, row=1, col=1)
fig.add_trace(area_trace, row=2, col=1)
fig.add_trace(peak_profile_trace, row=2, col=1)
for trace in time_span_traces:
    fig.add_trace(trace, row=2, col=2)

fig.update_xaxes(row=1, col=1, title_text="MZ [Th]")
fig.update_yaxes(row=1, col=1, title_text="Counts [a.u.]")

fig.update_xaxes(row=2, col=1, title_text="Time [s]")
fig.update_yaxes(row=2, col=1, title_text="Counts [a.u]")

fig.update_xaxes(row=2, col=2, title_text="MZ [Th]")
fig.update_yaxes(row=2, col=2, title_text="Counts [a.u.]")

# Adjust layout to prevent label overlap
fig.update_layout(
    height=500,
    margin=dict(t=30, b=30),
    title_x=0.5,
)

fig_widget = go.FigureWidget(fig)


def update_plots(target_mz_num):
    """Function to update the existing plot"""
    sum_trace, area_trace, peak_profile_trace, time_span_traces = create_traces(
        target_mz_num
    )

    with fig_widget.batch_update():
        fig_widget.data[0].x = sum_trace.x
        fig_widget.data[0].y = sum_trace.y

        fig_widget.data[1].x = area_trace.x
        fig_widget.data[1].y = area_trace.y

        fig_widget.data[2].x = peak_profile_trace.x
        fig_widget.data[2].y = peak_profile_trace.y

        for i, trace in enumerate(time_span_traces):
            fig_widget.data[3 + i].x = trace.x
            fig_widget.data[3 + i].y = trace.y


# Use interact to link the slider to the update function
slider = IntSlider(
    value=0,
    min=0,
    max=len(target_mzs) - 1,
    step=1,
    description="Target MZ Index",
    continuous_update=False,
)
ui = VBox([slider, fig_widget])
interact(update_plots, target_mz_num=slider)

display(ui)

In [ ]:
def interpolate_nan(arr):
    """
    Linearly interpolates NaN values in a NumPy array.

    Args:
        arr: A NumPy array.

    Returns:
        A new NumPy array with NaN values replaced using linear interpolation.
    """
    # Handle edge cases... (rest of the code remains the same)

    # Find indices of non-NaN values
    indices = np.flatnonzero(~np.isnan(arr))

    # Interpolate for NaN values between non-NaN indices
    return np.interp(np.arange(len(arr)), indices, arr[indices])

In [ ]:
# Get mz targets
with open("C:\\mascope_data\\targets\\20230510_stability.json") as _:
    target_isotopes = json.load(_)
list_of_mzs = [i["mz"] for i in target_isotopes["data"]]

mse, pear_r, comp_mzs = [], [], []
for i in list_of_mzs:
    # Peak heights calculated from areas
    heis = get_peak_heis(time_dict, i, precision=20)
    # if np.isnan(heis).any():
    #     continue
    # Get peak heights change in old way (as closest heigths from time spans)
    heis_old_fashioned = interpolate_nan(get_peak_profile(file, sum_peak_mzs, i))
    # Mean square error
    mse.append(np.mean((heis - heis_old_fashioned) ** 2))
    # Pearson correlation coefficient
    pear_r.append(pearsonr(heis, heis_old_fashioned)[0])
    # Add final mz
    comp_mzs.append(i)

In [ ]:
fig, (ax1, ax2) = plt.subplots(nrows=2, ncols=1, sharex=True, figsize=(7, 7))

ax1.scatter(comp_mzs, mse)
ax2.scatter(comp_mzs, pear_r)

ax1.set(
    xlabel="MZ [Th]",
    ylabel="Value [a.u.]",
    title="Mean square error",
    ylim=(min(mse), max(mse)),
)
ax2.set(
    xlabel="MZ [Th]", ylabel="Value [a.u.]", title="Pearson correlation coefficient"
)